<a href="https://colab.research.google.com/github/arctecnologia/InvestAI/blob/main/InvestAI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
!pip install -q fpdf2 matplotlib

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.0/81.0 kB 920.1 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.1/327.1 kB 3.7 MB/s eta 0:00:00


In [1]:
class InvestAI:
    def __init__(self, api_key):
        self.api_key = api_key
        self.nome = "InvestAI"

    def get_stock_data(self, ticker):
        """Busca dados em tempo real via BRAPI"""
        url = f"https://brapi.dev/api/quote/{ticker}?token={self.api_key}"
        # Lógica de request e tratamento (como no código anterior)
        pass

    def gerar_insight(self, dados):
        """Gera a resposta personalizada com a assinatura do agente"""
        if not dados:
            return f"🤖 Olá, aqui é o {self.nome}. Infelizmente, não consegui localizar os dados atualizados para este ativo na B3 no momento."

        preco = dados.get('regularMarketPrice')
        change = dados.get('regularMarketChangePercent')

        return f"🤖 [{self.nome} Analisa]: O ativo {dados['symbol']} está cotado a R${preco:.2f} ({change:.2f}% hoje). " \
               f"Com base na minha análise fundamentalista para 2026, noto que..."

In [6]:
!pip install -q "google-genai<2.0.0" "google-auth==2.47.0"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.7/52.7 kB 670.4 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.7/52.7 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.4/52.4 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.4/52.4 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.4/52.4 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.4/52.4 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.4/52.4 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.4/52.4 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.3/52.3 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 234.9/234.9 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 750.9/750.9 kB 8.4 MB/s eta 0:00:00


In [9]:
!pip install -q fpdf2 matplotlib yfinance

In [19]:
import requests
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import datetime
import yfinance as yf
from google import genai
from google.colab import userdata, files
from fpdf import FPDF

# ==========================================
# 1. Configurações Iniciais
# ==========================================
BRAPI_KEY = userdata.get('BRAPI_KEY')
GEMINI_KEY = userdata.get('GEMINI_KEY')
client = genai.Client(api_key=GEMINI_KEY)

PERIODOS_YF = {
    '1': ('1mo', '30 DIAS'),
    '2': ('3mo', '3 MESES'),
    '3': ('1y', '1 ANO'),
    '4': ('3y', '3 ANOS'),
    '5': ('5y', '5 ANOS')
}

class InvestAI:
    def __init__(self, brapi_key, gemini_client):
        self.brapi_key = brapi_key
        self.nome = "InvestAI"
        self.client = gemini_client
        self.modelo_nome = 'gemini-2.5-flash'

    def boas_vindas(self):
        return (
            f"🤖 Bem-vindo ao **{self.nome}**.\n"
            "Gerador de Relatórios Premium Híbrido impulsionado por IA.\n"
        )

    def descobrir_ticker(self, entrada):
        prompt = f"Apenas retorne o ticker oficial da B3 para a empresa: '{entrada}'. Sem explicações."
        try:
            return self.client.models.generate_content(model=self.modelo_nome, contents=prompt).text.strip().upper().replace(" ", "").replace(".", "")
        except:
            return entrada.upper()

    def get_kpis_exatos(self, ticker):
        """Busca os dados e resolve o bug do R$ 0.00 calculando o min/max manualmente"""
        try:
            acao = yf.Ticker(f"{ticker}.SA")
            info = acao.info
            hist_1y = acao.history(period="1y")

            valor_atual = info.get('currentPrice', info.get('regularMarketPrice', 0))
            dy = info.get('dividendYield', 0)

            # Cálculo robusto para evitar falhas da API
            min_52 = hist_1y['Low'].min() if not hist_1y.empty else 0
            max_52 = hist_1y['High'].max() if not hist_1y.empty else 0

            # Cálculo de Valorização de 12 Meses
            valorizacao_12m = 0
            if not hist_1y.empty and len(hist_1y) > 0:
                preco_antigo = hist_1y['Close'].iloc[0]
                preco_atual_hist = hist_1y['Close'].iloc[-1]
                valorizacao_12m = ((preco_atual_hist / preco_antigo) - 1) * 100

                # Se a API falhar no preço atual, pegamos do fechamento
                if valor_atual == 0: valor_atual = preco_atual_hist

            return {
                "valor_atual": valor_atual,
                "min_52": min_52,
                "max_52": max_52,
                "dy": (dy * 100) if dy else 0,
                "valorizacao_12m": valorizacao_12m,
                "nome_empresa": info.get('shortName', ticker)
            }
        except Exception:
            return None

    def get_historico_yf(self, ticker, periodo_yf):
        try:
            acao = yf.Ticker(f"{ticker}.SA")
            historico = acao.history(period=periodo_yf)
            return historico if not historico.empty else None
        except:
            return None

    def gerar_grafico_hibrido(self, ticker, historico, label_periodo):
        """Gráfico com as cores amadas (Azul/Laranja) e fundo limpo"""
        plt.style.use('default')
        fig, ax = plt.subplots(figsize=(10, 4.2), dpi=300)

        datas = historico.index
        precos = historico['Close']

        # Linha azul e área preenchida
        ax.plot(datas, precos, color='#1f77b4', linewidth=2.5, label='Preço de Fechamento')
        ax.fill_between(datas, precos, color='#1f77b4', alpha=0.15)

        # Média Móvel (Laranja)
        if len(precos) > 20:
            ma20 = precos.rolling(window=20).mean()
            ax.plot(datas, ma20, color='#ff7f0e', linewidth=1.5, linestyle='--', label='Média Móvel (20 dias)')

        # Formatação limpa do grid
        ax.yaxis.grid(True, linestyle='--', color='#e0e0e0', alpha=0.7)
        ax.xaxis.grid(False)

        ax.set_title(f'Performance Histórica - {ticker} ({label_periodo})', loc='center', fontsize=14, fontweight='bold', color='#333333', pad=15)

        # Limpando bordas (Spines)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.spines['left'].set_visible(False)

        ax.xaxis.set_major_formatter(mdates.DateFormatter('%d/%m/%Y' if label_periodo == '30 DIAS' else '%b/%Y'))
        plt.xticks(rotation=45, color='#555555', fontsize=9)
        plt.yticks(color='#555555', fontsize=9)

        # Legenda e Marcador final vermelho
        ax.legend(loc='upper left', frameon=True, facecolor='white', framealpha=0.9, fontsize=9)
        ultimo_preco = precos.iloc[-1]
        ax.scatter(datas[-1], ultimo_preco, color='red', s=45, zorder=5)
        ax.annotate(f' R$ {ultimo_preco:.2f}', (datas[-1], ultimo_preco), textcoords="offset points", xytext=(10,0), ha='left', va='center', fontweight='bold', color='red', fontsize=10)

        plt.tight_layout()
        caminho_imagem = f"grafico_{ticker}.png"
        plt.savefig(caminho_imagem)
        plt.close()
        return caminho_imagem

    def desenhar_card(self, pdf, x, y, titulo, valor, cor_fundo=None, cor_texto=(0,0,0), subtitulo=""):
        """Cards desenhados com títulos em PRETO absoluto para máxima legibilidade"""
        w, h = 45, 25
        if cor_fundo:
            pdf.set_fill_color(*cor_fundo)
            pdf.rect(x, y, w, h, style="F")
        else:
            pdf.set_draw_color(220, 220, 220)
            pdf.line(x+w, y+2, x+w, y+h-2)

        # Título
        pdf.set_xy(x, y + 4)
        pdf.set_font("helvetica", "B", 7)
        if cor_fundo:
            pdf.set_text_color(255, 255, 255) # Branco se tiver fundo azul
        else:
            pdf.set_text_color(0, 0, 0) # Preto absoluto nos demais
        pdf.cell(w, 5, titulo, align="C")

        # Valor
        pdf.set_xy(x, y + 10)
        pdf.set_font("helvetica", "B", 13)
        pdf.set_text_color(*cor_texto)
        pdf.cell(w, 8, valor, align="C")

        # Subtítulo
        if subtitulo:
            pdf.set_xy(x, y + 18)
            pdf.set_font("helvetica", "", 7)
            if cor_fundo:
                pdf.set_text_color(255, 255, 255) # Branco se tiver fundo azul
            else:
                pdf.set_text_color(0, 0, 0) # Preto absoluto nos demais
            pdf.cell(w, 4, subtitulo, align="C")

    def exportar_pdf_premium(self, ticker, analise, caminho_grafico, kpis):
        pdf = FPDF()
        pdf.add_page()

        # ==========================================
        # HEADER
        # ==========================================
        pdf.set_fill_color(31, 119, 180) # Azul Profissional
        pdf.set_text_color(255, 255, 255)
        pdf.set_font("helvetica", "B", 18)
        pdf.cell(0, 15, f" RELATÓRIO EXECUTIVO: {ticker} ", border=0, new_x="LMARGIN", new_y="NEXT", align="C", fill=True)

        pdf.set_text_color(100, 100, 100)
        pdf.set_font("helvetica", "I", 10)
        pdf.cell(0, 8, f"Gerado pelo assistente InvestAI | Data: {datetime.now().strftime('%d/%m/%Y')}", align="C", new_x="LMARGIN", new_y="NEXT")
        pdf.ln(5)

        # ==========================================
        # PAINEL DE INDICADORES CARDS
        # ==========================================
        y_cards = pdf.get_y()

        # CARD 1: Valor Atual
        self.desenhar_card(pdf, 10, y_cards, "VALOR ATUAL", f"R$ {kpis['valor_atual']:.2f}", cor_fundo=(0, 84, 131), cor_texto=(255,255,255))

        # CARD 2: Min e Max
        self.desenhar_card(pdf, 55, y_cards, "MIN / MÁX (52 SEM.)", f"R$ {kpis['min_52']:.2f}", cor_texto=(0,0,0), subtitulo=f"MÁX: R$ {kpis['max_52']:.2f}")

        # CARD 3: Dividend Yield
        self.desenhar_card(pdf, 100, y_cards, "DIVIDEND YIELD", f"{kpis['dy']:.2f}%", cor_texto=(0,0,0), subtitulo="ÚLTIMOS 12 MESES")

        # CARD 4: Valorização
        cor_val = (0, 150, 0) if kpis['valorizacao_12m'] >= 0 else (200, 0, 0)
        sinal = "+" if kpis['valorizacao_12m'] >= 0 else ""
        self.desenhar_card(pdf, 145, y_cards, "VALORIZAÇÃO (12M)", f"{sinal}{kpis['valorizacao_12m']:.2f}%", cor_texto=cor_val)

        pdf.set_y(y_cards + 30)

        # ==========================================
        # GRÁFICO E TEXTO IA
        # ==========================================
        if caminho_grafico:
            pdf.image(caminho_grafico, x=5, w=195)
            pdf.ln(2)

        pdf.set_font("helvetica", size=10)
        pdf.set_text_color(50, 50, 50)

        try:
            pdf.multi_cell(0, 5, analise, markdown=True)
        except:
            pdf.multi_cell(0, 5, analise.replace('**', '').replace('*', ''))

        # Rodapé
        pdf.ln(8)
        pdf.set_font("helvetica", "B", 8)
        pdf.set_text_color(31, 119, 180)
        pdf.cell(0, 10, "by Lilicatech & arctecnologia", align="R")

        nome_arquivo = f"InvestAI_{ticker}_Premium.pdf"
        pdf.output(nome_arquivo)
        return nome_arquivo

    def analisar_ativo(self, entrada, opcao_periodo):
        periodo_yf, label_periodo = PERIODOS_YF.get(opcao_periodo, ('1y', '1 ANO'))

        print(f"\n🔍 [{self.nome}]: Identificando o ativo exato...")
        ticker = self.descobrir_ticker(entrada)

        print(f"🔄 [{self.nome}]: Coletando KPIs e histórico de preços...")
        kpis = self.get_kpis_exatos(ticker)
        historico_dados = self.get_historico_yf(ticker, periodo_yf)

        if not kpis or historico_dados is None:
            return f"❌ [{self.nome}]: Dados de {ticker} indisponíveis no momento."

        print(f"📊 [{self.nome}]: Desenhando gráfico avançado...")
        caminho_grafico = self.gerar_grafico_hibrido(ticker, historico_dados, label_periodo)

        print(f"🧠 [{self.nome}]: Processando o cenário com IA Generativa...")

        prompt = (
            f"Atue como um analista financeiro sênior avaliando {ticker}.\n"
            f"Preço: R${kpis['valor_atual']:.2f} | DY: {kpis['dy']:.2f}% | Val. 12M: {kpis['valorizacao_12m']:.2f}%.\n"
            f"Escreva 3 parágrafos diretos e executivos contendo: "
            f"1) Visão do Preço Atual e Valorização; 2) Saúde de Dividendos; 3) Perspectiva Setorial para 2026. "
            f"Use formatação markdown (**negritos**)."
        )

        resposta_ia = self.client.models.generate_content(model=self.modelo_nome, contents=prompt).text
        print("\n" + resposta_ia + "\n" + "-"*60)

        print(f"📑 [{self.nome}]: Compilando painel premium em PDF...")
        arquivo_pdf = self.exportar_pdf_premium(ticker, resposta_ia, caminho_grafico, kpis)
        files.download(arquivo_pdf)

        return "✅ Relatório baixado com sucesso! Confira o design Híbrido."

# ==========================================
# 3. Interface de Chat
# ==========================================
assistente = InvestAI(BRAPI_KEY, client)
print(assistente.boas_vindas())

while True:
    entrada = input("\n👤 Digite o ativo (ex: BEEF3) ou 'sair': ").strip()
    if entrada.upper() in ['SAIR', 'EXIT', 'FIM']: break
    if not entrada: continue

    print("\n⏱️ Escolha o filtro do Gráfico:")
    print("[1] 30 Dias | [2] 3 Meses | [3] 1 Ano (Padrão) | [4] 3 Anos | [5] 5 Anos")
    opcao_periodo = input("Digite o número (ou pressione Enter para o Padrão): ").strip()

    print(assistente.analisar_ativo(entrada, opcao_periodo))

🤖 Bem-vindo ao **InvestAI**.
Gerador de Relatórios Premium Híbrido impulsionado por IA.


👤 Digite o ativo (ex: BEEF3) ou 'sair': BEEF3

⏱️ Escolha o filtro do Gráfico:
[1] 30 Dias | [2] 3 Meses | [3] 1 Ano (Padrão) | [4] 3 Anos | [5] 5 Anos
Digite o número (ou pressione Enter para o Padrão): 1

🔍 [InvestAI]: Identificando o ativo exato...
🔄 [InvestAI]: Coletando KPIs e histórico de preços...
📊 [InvestAI]: Desenhando gráfico avançado...
🧠 [InvestAI]: Processando o cenário com IA Generativa...

Prezado(a) investidor(a),

Com base nos dados apresentados para **BEEF3**, segue uma avaliação executiva:

O preço atual de **R$4.12** para BEEF3 reflete um período de intensa pressão e volatilidade, com uma **desvalorização de 23.63% nos últimos 12 meses**. Esta performance sugere que o mercado está precificando desafios significativos, sejam eles operacionais da empresa ou macroeconômicos e setoriais. É imperativo uma análise fundamentalista aprofundada para determinar se o valuation atual repr

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Relatório baixado com sucesso! Confira o design Híbrido.

👤 Digite o ativo (ex: BEEF3) ou 'sair': sair
